# Yelp Shared Split Generator

**Run this notebook once before any training notebook.**

It:
1. Loads all four Yelp datasets (YelpZip, YelpChi ×2, YelpNYC) with rating + date metadata
2. **Removes collection-duplicate reviews** — the same physical review copied across overlapping source files (YelpNYC ⊂ YelpZip). Dedup key = (normalized text, rating, date). Natural duplicates (different reviewers writing the same short text like "good") are preserved, since they differ in rating/date.
3. Shuffles and balances to a 1:1 fake/genuine ratio
4. Creates one shared stratified **80/10/10 split** (grouped by normalized text → zero leakage)
5. Saves it to `yelp_split.csv` on your Google Drive
6. Checks for **cross-split duplicate text leakage**

All training notebooks (sklearn, LSTM, DistilBERT) load from this CSV,
guaranteeing every model trains, validates, and tests on the **exact same rows**.

---
### Instructions
1. Upload the following files to your Google Drive:
   - `YelpZip/metadata` and `YelpZip/reviewContent`
   - `YelpChi/output_meta_yelpResData_NRYRcleaned.txt` and `output_review_...`
   - `YelpChi/output_meta_yelpHotelData_NRYRcleaned.txt` and `output_review_...`
   - `YelpNYC/metadata` and `YelpNYC/reviewContent`
2. Update the paths in **cell 2** if needed.
3. Run all cells in order.
4. `yelp_split.csv` will be saved to your Google Drive automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THESE PATHS if your files are inside a subfolder ──
YELPZIP_META    = '/content/drive/MyDrive/YelpZip/metadata'
YELPZIP_REVIEWS = '/content/drive/MyDrive/YelpZip/reviewContent'

YELPCHI_META_RES    = '/content/drive/MyDrive/YelpChi/output_meta_yelpResData_NRYRcleaned.txt'
YELPCHI_REVIEW_RES  = '/content/drive/MyDrive/YelpChi/output_review_yelpResData_NRYRcleaned.txt'
YELPCHI_META_HOT    = '/content/drive/MyDrive/YelpChi/output_meta_yelpHotelData_NRYRcleaned.txt'
YELPCHI_REVIEW_HOT  = '/content/drive/MyDrive/YelpChi/output_review_yelpHotelData_NRYRcleaned.txt'

YELPNYC_META    = '/content/drive/MyDrive/YelpNYC/metadata'
YELPNYC_REVIEWS = '/content/drive/MyDrive/YelpNYC/reviewContent'

# Output — shared by all training notebooks
SPLIT_CSV = '/content/drive/MyDrive/yelp_split.csv'

In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
def load_yelpzip(meta_path, review_path):
    """YelpZip / YelpNYC.
    metadata     : user_id  prod_id  rating  label  date   (tab-separated)
    reviewContent: user_id  prod_id  date    text          (tab-separated)
    Returns parallel lists: texts, labels, ratings, dates.
    """
    texts, labels, ratings, dates = [], [], [], []
    with open(meta_path, encoding='utf-8', errors='ignore') as mf, \
         open(review_path, encoding='utf-8', errors='ignore') as rf:
        for meta_line, review_line in zip(mf, rf):
            m = meta_line.strip().split('\t')
            r = review_line.strip().split('\t')
            if len(m) < 5 or len(r) < 4:
                continue
            label_str = m[3]
            if label_str == '-1':
                label = 1
            elif label_str == '1':
                label = 0
            else:
                continue
            text = r[3].strip()
            if text:
                texts.append(text)
                labels.append(label)
                ratings.append(m[2])   # star rating
                dates.append(m[4])     # review date (YYYY-MM-DD)
    return texts, labels, ratings, dates


def load_yelpchi(meta_path, review_path):
    """YelpChi.
    metadata: date  reviewID  reviewerID  productID  label(N/Y)  ...  rating(last col)
    review  : one review per line (the full text).
    Returns parallel lists: texts, labels, ratings, dates.
    """
    texts, labels, ratings, dates = [], [], [], []
    with open(meta_path, encoding='utf-8', errors='ignore') as mf, \
         open(review_path, encoding='utf-8', errors='ignore') as rf:
        for meta_line, review_line in zip(mf, rf):
            parts = meta_line.strip().split()
            if len(parts) < 5:
                continue
            label_str = parts[4]
            if label_str == 'Y':
                label = 1
            elif label_str == 'N':
                label = 0
            else:
                continue
            text = review_line.strip()
            if text:
                texts.append(text)
                labels.append(label)
                ratings.append(parts[8] if len(parts) > 8 else '')  # rating (last col)
                dates.append(parts[0])                              # review date (M/D/YYYY)
    return texts, labels, ratings, dates

In [ ]:
import re

def normalize_text(t):
    """Lowercase + collapse whitespace — for grouping/dedup, not saved to CSV."""
    return re.sub(r'\s+', ' ', t.strip().lower())

print('Loading YelpZip ...')
zip_texts, zip_labels, zip_ratings, zip_dates = load_yelpzip(YELPZIP_META, YELPZIP_REVIEWS)
print(f'  {len(zip_texts):,} reviews')

print('Loading YelpChi (restaurants) ...')
chi_res_texts, chi_res_labels, chi_res_ratings, chi_res_dates = load_yelpchi(YELPCHI_META_RES, YELPCHI_REVIEW_RES)
print(f'  {len(chi_res_texts):,} reviews')

print('Loading YelpChi (hotels) ...')
chi_hot_texts, chi_hot_labels, chi_hot_ratings, chi_hot_dates = load_yelpchi(YELPCHI_META_HOT, YELPCHI_REVIEW_HOT)
print(f'  {len(chi_hot_texts):,} reviews')

print('Loading YelpNYC ...')
nyc_texts, nyc_labels, nyc_ratings, nyc_dates = load_yelpzip(YELPNYC_META, YELPNYC_REVIEWS)
print(f'  {len(nyc_texts):,} reviews')

# ── Combine all sources ───────────────────────────────────────────────────────
df_all = pd.DataFrame({
    'text':   zip_texts + chi_res_texts + chi_hot_texts + nyc_texts,
    'label':  zip_labels + chi_res_labels + chi_hot_labels + nyc_labels,
    'rating': zip_ratings + chi_res_ratings + chi_hot_ratings + nyc_ratings,
    'date':   zip_dates + chi_res_dates + chi_hot_dates + nyc_dates,
})
print(f'\nTotal loaded (all sources): {len(df_all):,} reviews')

# ── Remove collection-duplicate reviews ───────────────────────────────────────
# The SAME physical review appears in multiple source files — YelpNYC is a
# geographic subset of YelpZip, so NYC reviews appear verbatim in both. Those
# copies share identical (normalized text, rating, date), so we drop them and
# keep ONE copy.
#
# "Natural" duplicates — different reviewers who independently wrote the same
# short text (e.g. "good", "great food") — almost always differ in rating
# and/or date, so they are PRESERVED as distinct rows. We deliberately do NOT
# key on user_id / prod_id because each dataset uses its own ID numbering, so a
# cross-dataset copy would otherwise look distinct.
df_all['text_norm'] = df_all['text'].apply(normalize_text)
before = len(df_all)
df_all = df_all.drop_duplicates(subset=['text_norm', 'rating', 'date'], keep='first').reset_index(drop=True)
removed = before - len(df_all)
print(f'Removed {removed:,} collection-duplicate reviews ({removed/before:.1%}) '
      f'— same review copied across source files')
print(f'Remaining unique reviews: {len(df_all):,}')

all_texts  = df_all['text'].tolist()
all_labels = df_all['label'].tolist()

# Shuffle
combined = list(zip(all_texts, all_labels))
random.seed(SEED)
random.shuffle(combined)
all_texts, all_labels = zip(*combined)
all_texts  = list(all_texts)
all_labels = list(all_labels)

n_fake    = sum(all_labels)
n_genuine = len(all_labels) - n_fake
print(f'\nTotal (before balancing): {len(all_texts):,} reviews')
print(f'  Fake   : {n_fake:,}  ({n_fake/len(all_labels):.1%})')
print(f'  Genuine: {n_genuine:,}  ({n_genuine/len(all_labels):.1%})')

# Balanced sampling — keep all fake + equal number of genuine
fake_idx    = [i for i, l in enumerate(all_labels) if l == 1]
genuine_idx = [i for i, l in enumerate(all_labels) if l == 0]
rng = random.Random(SEED)
genuine_sample = rng.sample(genuine_idx, len(fake_idx))
balanced_idx   = fake_idx + genuine_sample
rng.shuffle(balanced_idx)
all_texts  = [all_texts[i]  for i in balanced_idx]
all_labels = [all_labels[i] for i in balanced_idx]
print(f'\nAfter balancing: {len(all_texts):,} reviews (1:1 ratio)')

# ── Grouped split on normalized text ─────────────────────────────────────────
# Even after dedup, identical short texts written by different reviewers (natural
# duplicates) remain. We split at the unique-normalized-text level so every
# occurrence of a text goes to the same split → zero train/val/test leakage.
df = pd.DataFrame({'text': all_texts, 'label': all_labels})
df['text_norm'] = df['text'].apply(normalize_text)

# ── Mixed-label groups ────────────────────────────────────────────────────────
label_diversity = df.groupby('text_norm')['label'].nunique()
n_unique_groups = len(label_diversity)
n_mixed         = (label_diversity > 1).sum()
print(f'\nUnique text groups (normalized): {n_unique_groups:,}')
print(f'Groups with conflicting labels : {n_mixed:,}', end='')
if n_mixed > 0:
    print(f'  ({n_mixed / n_unique_groups:.2%} of groups) — dominant label used for split assignment')
else:
    print(' — none found')

# ── Text-level stratified 80/10/10 split ─────────────────────────────────────
text_label_df = (df.groupby('text_norm')['label']
                 .agg(lambda x: x.mode()[0])
                 .reset_index()
                 .rename(columns={'label': 'dominant_label'}))

tv_norms, te_norms, tv_labels, te_labels = train_test_split(
    text_label_df['text_norm'].values,
    text_label_df['dominant_label'].values,
    test_size=0.1, random_state=SEED,
    stratify=text_label_df['dominant_label'].values,
)
tr_norms, va_norms, tr_labels, va_labels = train_test_split(
    tv_norms, tv_labels,
    test_size=1/9, random_state=SEED, stratify=tv_labels,
)

norm_to_split = {}
for t in tr_norms: norm_to_split[t] = 'train'
for t in va_norms: norm_to_split[t] = 'val'
for t in te_norms: norm_to_split[t] = 'test'
df['split'] = df['text_norm'].map(norm_to_split)

# ── Class balance per split ───────────────────────────────────────────────────
print(f'\n{"Split":<8} {"Rows":>8}  {"Genuine":>9}  {"Fake":>8}  {"Fake%":>6}')
print('-' * 46)
for s in ('train', 'val', 'test'):
    sub      = df[df['split'] == s]
    n_g      = (sub['label'] == 0).sum()
    n_f      = (sub['label'] == 1).sum()
    pct_fake = n_f / len(sub)
    print(f'{s:<8} {len(sub):>8,}  {n_g:>9,}  {n_f:>8,}  {pct_fake:>6.1%}')

# Save original text (not normalized) to CSV
df[['text', 'label', 'split']].to_csv(SPLIT_CSV, index=False)
print(f'\nSaved: {SPLIT_CSV}  ({len(df):,} rows)')

In [ ]:
print('=== Leakage Verification ===')
print()

# Use the same normalization as the split to get an accurate answer.
# Checking raw text would miss case/whitespace variants that were grouped together.
train_norms = set(df[df['split'] == 'train']['text_norm'])
val_norms   = set(df[df['split'] == 'val'  ]['text_norm'])
test_norms  = set(df[df['split'] == 'test' ]['text_norm'])

# Cross-split overlaps at the normalized-text level
tv_overlap = len(train_norms & val_norms)
tt_overlap = len(train_norms & test_norms)
vt_overlap = len(val_norms   & test_norms)
print(f'Unique normalized text groups per split:')
print(f'  Train : {len(train_norms):,}')
print(f'  Val   : {len(val_norms):,}')
print(f'  Test  : {len(test_norms):,}')
print()
print(f'Cross-split text-group overlaps (should all be 0):')
print(f'  Train ∩ Val  : {tv_overlap:,}')
print(f'  Train ∩ Test : {tt_overlap:,}')
print(f'  Val  ∩ Test  : {vt_overlap:,}')
print()

# Actual leaked rows
test_leaked = df[(df['split'] == 'test') & (df['text_norm'].isin(train_norms))]
val_leaked  = df[(df['split'] == 'val')  & (df['text_norm'].isin(train_norms))]
n_test      = (df['split'] == 'test').sum()
n_val       = (df['split'] == 'val').sum()

test_leak_pct = len(test_leaked) / n_test
val_leak_pct  = len(val_leaked)  / n_val
print(f'Actual leaked test rows : {len(test_leaked):,}  ({test_leak_pct:.3%} of test set)')
print(f'Actual leaked val  rows : {len(val_leaked):,}  ({val_leak_pct:.3%} of val set)')
print()

if tt_overlap == 0 and tv_overlap == 0:
    print('PASS — No train→val or train→test leakage. Training pipeline is clean.')
else:
    print('WARNING — Leakage detected. Do not proceed to training.')
    print('          Consider re-examining the normalization or grouping logic.')

## Next steps
`yelp_split.csv` is saved to your Google Drive. Run the three training notebooks in any order:
- `train_yelp_sklearn.ipynb`
- `train_yelp_lstm.ipynb`
- `train_yelp_distilbert.ipynb`

Each loads directly from `yelp_split.csv` — no raw dataset files needed.